# Results handling

This worked example shows how to work with Breathe Simulate results objects, from quick equilibrium KPIs to dynamic simulations and batch runs.

You’ll learn to master:
- Fetch equilibrium KPIs for a baseline cell
- Inspect and compare designs
- Run a worked example simulation with a `Cycler`
- Explore dynamic data and plot key variables
- Scale up to batch simulations and summarize results


## Setup  
<a id="setup"></a>

This imports the Breathe Simulate API and cycler package used throughout the worked example


In [1]:
from breathe_simulate import api_interface as api, Cycler

The first result we can get from the api is the equilibrium KPIs


## Designs & Equilibrium KPIs  

We start with a **baseline cell** (here: `"Molicel P45B"`).  
`get_eqm_kpis` returns a results object with convenient accessors for capacity and other KPIs.



In [2]:
eqm_results = api.get_eqm_kpis("Molicel P45B")

Submitted run_sim job b868326c-318e-4253-8fc3-593321adfcc9


The results class is


In [3]:
eqm_results.__class__

breathe_simulate.results.SingleSimulationResults

With methods and attributes


In [4]:
for attr in dir(eqm_results):
    if not attr.startswith("_"):
        print(f"{attr}")

baseline
capacity
compare_designs
design_names
design_parameters
dynamic_data
dynamic_kpis
get_dynamic_data
get_input_parameters
get_kpis
get_normalized_kpis
get_summary
kpis
np_adjusted
plot_dynamic_response
plot_radar
plot_sensitivities
plot_voltage_response
print_dynamic_variables


`baseline` provides a shortcut to the Baseline KPIs


In [5]:
eqm_results.baseline

KPI
Capacity [Ah]                                   4.500000
Nominal Voltage [V]                             3.656630
Energy [Wh]                                    16.454834
Gravimetric Energy Density [Wh/kg]            241.196572
Volumetric Energy Density [Wh/l]              665.310310
Minimum Anode Voltage [mV]                     80.487709
Weight [g]                                     68.221675
Volume [l]                                      0.024733
Specific Heat Capacity [kJ/kg/K]                1.200000
Thermal Conductivity In-plane [W/m/K]          42.235526
Thermal Conductivity Through-plane [W/m/K]      1.085540
Price [USD]                                     1.157604
Price [USD/kWh]                                70.350411
Price [USD/kg]                                 16.968278
Name: Baseline, dtype: float64

`capacity` gets the capacity value for the Baseline design


In [6]:
eqm_results.capacity

np.float64(4.4999999999999964)

`compare_designs` compares the KPIs of the designs but will return an error if only the Baseline design is present


In [7]:
try:
    eqm_results.compare_designs()
except ValueError as e:
    print(e)

Only Baseline design found. Cannot compare designs.


`design_names` gets list of design names.


In [8]:
eqm_results.design_names

['Baseline']

`dynamic_data` returns the dynamic data as result of passing a cycler to the api and running a simulation for each design and input parameter. However, in this case it will be empty as no designs or cycler objects were passed.


In [9]:
eqm_results.dynamic_data

{}

`get_input_parameters` gets the original input parameters used for the api call.


In [10]:
eqm_results.get_input_parameters()

{'base_battery': 'Molicel P45B', 'designs': [], 'formats': []}

`get_kpis` and `kpis` both return the kpi data. `get_kpis` can accept a design name and only return data for that design providing it exists.


In [11]:
try:
    eqm_results.get_kpis("testDesign")
except ValueError as e:
    print(e)

Design(s) not found: ['testDesign']. Available designs: ['Baseline'].


`get_normalized_kpis` divides the values for each design by the Baseline values. If there is only a Baseline design this will return all 1.0s.


In [12]:
eqm_results.get_normalized_kpis()

,Baseline
KPI,
Capacity [Ah],1.0
Nominal Voltage [V],1.0
Energy [Wh],1.0
Gravimetric Energy Density [Wh/kg],1.0
Volumetric Energy Density [Wh/l],1.0
Minimum Anode Voltage [mV],1.0
Weight [g],1.0
Volume [l],1.0
Specific Heat Capacity [kJ/kg/K],1.0


`get_summary` prints a summary of the results


In [13]:
eqm_results.get_summary()

{'is_batch': False,
 'num_simulations': 1,
 'design_names': ['Baseline'],
 'num_designs': 1,
 'input_parameters': {'base_battery': 'Molicel P45B',
  'designs': [],
  'formats': []}}

`plot_sensitivities` plots the sensitivities of each KPI to variation in each design parameter


In [14]:
eqm_results.plot_sensitivities()

Some of the methods will not work without designs or dynamic data generated from passing a cycler to the api


In [15]:
try:
    eqm_results.plot_radar()
except ValueError as e:
    print(e)

Only Baseline design found. Cannot compare designs.

In [16]:
try:
    eqm_results.plot_dynamic_response()
except ValueError as e:
    print(e)

No dynamic data available for this simulation


In [17]:
try:
    eqm_results.plot_voltage_response()
except ValueError as e:
    print(e)

No dynamic data available for this simulation


In [18]:
try:
    eqm_results.print_dynamic_variables()
except ValueError as e:
    print(e)

No dynamic data available for this simulation


## Designs & Dynamic Data  
<a id="designs--dynamic-data"></a>

Let's add a virtual cell design and run a dynamic simulation in parallel with our baseline cell. In this example we will adjust the NPratio to a fixed value of 1.0.


In [19]:
designs = [{"designName": "Lower NP", "NPratio": 1.0}]

### Build a Cycler

A `Cycler` describes the protocol (e.g., CC charge to a voltage limit).  
Here we set units and (optionally) pass in the baseline capacity for relative C‑rates.


In [20]:
cycler = Cycler(selected_unit="C", cell_capacity=eqm_results.capacity)

### Worked Example: P45B Charge Run  
<a id="worked-example-p45b-charge-run"></a>

We’ll run **one** simulation using our cycler and a simple design list (including Baseline).  
This returns a results object with:
- `.kpis` and `.get_kpis(designName)` for design‑specific KPIs  
- `.compare_designs()` to compare across designs  
- `.plot_*` helpers for quick visualizations


In [21]:
single_output = api.run_sim(
    base_battery="Molicel P45B",
    cycler=cycler.cccv(1.0, -1.0, 0.01, 4.2, 2.6, 60.0, 60.0),
    designs=designs,
    initialSoC=0.5,
    initialTemperature_degC=25.0,
    ambientTemperature_degC=25.0,
)

Running...:   0%|          | 0/1 [00:00<?, ? designs/s]

Submitted run_sim job 871223ba-e921-471d-9ce4-d9573f5608d4


Running...: 100%|██████████| 1/1 [00:08<00:00,  8.74s/ designs]

Running...: 100%|██████████| 1/1 [00:08<00:00,  8.74s/ designs]

In [22]:
single_output.kpis

,Baseline,Lower NP
KPI,,
Capacity [Ah],4.500000,4.626562
Nominal Voltage [V],3.656630,3.659476
Energy [Wh],16.454834,16.930793
Gravimetric Energy Density [Wh/kg],241.196572,246.933259
Volumetric Energy Density [Wh/l],665.310310,684.554503
Minimum Anode Voltage [mV],80.487709,77.406754
Weight [g],68.221675,68.564246
Volume [l],0.024733,0.024733
Specific Heat Capacity [kJ/kg/K],1.200000,1.197895


If we only want the equilibrium KPIs for a specific design, we can use `.get_kpis(designName)`

In [23]:
single_output.get_kpis("Lower NP")

KPI
Capacity [Ah]                                   4.626562
Nominal Voltage [V]                             3.659476
Energy [Wh]                                    16.930793
Gravimetric Energy Density [Wh/kg]            246.933259
Volumetric Energy Density [Wh/l]              684.554503
Minimum Anode Voltage [mV]                     77.406754
Weight [g]                                     68.564246
Volume [l]                                      0.024733
Specific Heat Capacity [kJ/kg/K]                1.197895
Thermal Conductivity In-plane [W/m/K]          42.847217
Thermal Conductivity Through-plane [W/m/K]      1.074846
Price [USD]                                     1.164483
Price [USD/kWh]                                68.779013
Price [USD/kg]                                 16.983826
Name: Lower NP, dtype: float64

We can now compare the results of our virtual cell design with the baseline cell.

In [24]:
single_output.compare_designs()

In [25]:
single_output.plot_voltage_response()

In [26]:
single_output.print_dynamic_variables()

Available variables:
Time [s]
Current [A]
Charge current [A]
Discharge current [A]
Voltage [V]
Anode voltage [V]
Anode-Separator Reference Potential [mV]
Cathode voltage [V]
Cell open-circuit voltage [V]
Negative electrode OCP with hysteresis [V]
Positive electrode OCP with hysteresis [V]
SoC
Average negative particle stoichiometry
Average positive particle stoichiometry
Cell temperature [°C]
Heat generation total [W]


In [27]:
single_output.plot_dynamic_response("Cell temperature [°C]")

## Batch Results  
<a id="batch-results"></a>

Now we scale up: pass **lists** (e.g., `initialSoC`) to evaluate multiple starting conditions in one call.

- Use `.get_summary()` for an at‑a‑glance table across runs  
- Use `.get_dynamic_data(design)` to retrieve time‑series data  
- Use `.plot_dynamic_response(variable)` to visualize across designs/runs


In [28]:
initial_socs = [0.1, 0.25, 0.5, 0.75, 0.9]
batch_output = api.run_sim(
    base_battery="Molicel P45B",
    cycler=cycler.cc_chg(1.0, 4.2),
    designs=designs,
    initialSoC=initial_socs,  # List of SOC values
    initialTemperature_degC=25.0,
    ambientTemperature_degC=25.0,
)

Running...:   0%|          | 0/1 [00:00<?, ? designs/s]

Running...:   0%|          | 0/1 [00:00<?, ? designs/s]

Running...:   0%|          | 0/1 [00:00<?, ? designs/s]

Running...:   0%|          | 0/1 [00:00<?, ? designs/s]

Submitted run_sim job b2ceabd4-f4eb-4afc-82e4-0395fa23feb5


Submitted run_sim job 02f978af-4e7c-4bce-bb6f-031d7147abbd


... (10 outputs left out)

Running...: 100%|██████████| 1/1 [00:14<00:00, 14.77s/ designs]

Running...: 100%|██████████| 1/1 [00:14<00:00, 14.78s/ designs]

Running...: 100%|██████████| 1/1 [00:11<00:00, 11.86s/ designs]

Running...: 100%|██████████| 1/1 [00:11<00:00, 11.86s/ designs]

Here we get a different class, but the methods are very similar.


In [29]:
batch_output.__class__

breathe_simulate.results.BatchSimulationResults

In [30]:
for attr in dir(batch_output):
    if not attr.startswith("_"):
        print(f"{attr}")

baseline
capacity
compare_designs
design_names
dynamic_data
dynamic_kpis
first
get_dynamic_data
get_input_parameters
get_kpis
get_normalized_kpis
get_summary
num_simulations
plot_dynamic_response
plot_radar
plot_sensitivities
plot_voltage_response
print_dynamic_variables


In [31]:
batch_output.get_summary()

{'is_batch': True,
 'num_simulations': 5,
 'design_names': ['Baseline', 'Lower NP'],
 'num_designs': 2,
 'input_parameters': [{'base_battery': 'Molicel P45B',
   'virtual_battery_uuid': None,
   'cycler': {'cycle_type': 'CC_CHG',
    'selected_unit': 'C',
    'control_parameters': {'I_chg': 1.0, 'V_max': 4.2}},
   'designs': [{'designName': 'Lower NP', 'NPratio': 1.0}],
   'socCell_0': 0.1,
   'initialTemperature_degC': 25.0,
   'ambientTemperature_degC': 25.0,
   'heatTransferCoefficient': 35.0,
   'isothermal': False,
   'initialOCVType': None},
  {'base_battery': 'Molicel P45B',
   'virtual_battery_uuid': None,
   'cycler': {'cycle_type': 'CC_CHG',
    'selected_unit': 'C',
    'control_parameters': {'I_chg': 1.0, 'V_max': 4.2}},
   'designs': [{'designName': 'Lower NP', 'NPratio': 1.0}],
   'socCell_0': 0.25,
   'initialTemperature_degC': 25.0,
   'ambientTemperature_degC': 25.0,
   'heatTransferCoefficient': 35.0,
   'isothermal': False,
   'initialOCVType': None},
  {'base_batter

In [32]:
batch_output.get_kpis()

,Baseline,Lower NP
KPI,,
Capacity [Ah],4.500000,4.626562
Nominal Voltage [V],3.656630,3.659476
Energy [Wh],16.454834,16.930793
Gravimetric Energy Density [Wh/kg],241.196572,246.933259
Volumetric Energy Density [Wh/l],665.310310,684.554503
Minimum Anode Voltage [mV],80.487709,77.406754
Weight [g],68.221675,68.564246
Volume [l],0.024733,0.024733
Specific Heat Capacity [kJ/kg/K],1.200000,1.197895


We can plot the variables of interest for the designs across the batch

In [33]:
batch_output.plot_dynamic_response("Voltage [V]")

In [34]:
batch_output.plot_dynamic_response("Cell temperature [°C]")

We can also retrieve all of the dynamic data for the batch for all designs, or for a single design


In [35]:
batch_output.dynamic_data

[{'Baseline': {'Time [s]': [0.0,
    3.0946945571962505,
    6.189389114392501,
    9.284083671588752,
    12.378778228785002,
    15.473472785981253,
    18.568167343177503,
    21.662861900373755,
    24.757556457570004,
    27.852251014766253,
    30.946945571962505,
    34.04164012915876,
    37.136334686355006,
    40.231029243551255,
    43.32572380074751,
    46.42041835794376,
    49.51511291514001,
    52.60980747233626,
    55.704502029532506,
    58.79919658672876,
    61.89389114392501,
    64.98858570112127,
    68.08328025831752,
    71.17797481551376,
    74.27266937271001,
    77.36736392990626,
    80.46205848710251,
    83.55675304429876,
    86.65144760149502,
    89.74614215869127,
    92.84083671588752,
    95.93553127308377,
    99.03022583028002,
    102.12492038747627,
    105.21961494467251,
    108.31430950186876,
    111.40900405906501,
    114.50369861626127,
    117.59839317345752,
    120.69308773065377,
    123.78778228785002,
    126.88247684504627,
    

In [36]:
batch_output.get_dynamic_data("Baseline")

[{'Baseline': {'Time [s]': [0.0,
    3.0946945571962505,
    6.189389114392501,
    9.284083671588752,
    12.378778228785002,
    15.473472785981253,
    18.568167343177503,
    21.662861900373755,
    24.757556457570004,
    27.852251014766253,
    30.946945571962505,
    34.04164012915876,
    37.136334686355006,
    40.231029243551255,
    43.32572380074751,
    46.42041835794376,
    49.51511291514001,
    52.60980747233626,
    55.704502029532506,
    58.79919658672876,
    61.89389114392501,
    64.98858570112127,
    68.08328025831752,
    71.17797481551376,
    74.27266937271001,
    77.36736392990626,
    80.46205848710251,
    83.55675304429876,
    86.65144760149502,
    89.74614215869127,
    92.84083671588752,
    95.93553127308377,
    99.03022583028002,
    102.12492038747627,
    105.21961494467251,
    108.31430950186876,
    111.40900405906501,
    114.50369861626127,
    117.59839317345752,
    120.69308773065377,
    123.78778228785002,
    126.88247684504627,
    